In [6]:
import os 
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

In [7]:
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, ModelSettings

In [12]:
local_model = OpenAIChatCompletionsModel(
    model = "llama3.2",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url= "http://localhost:11434/v1"
    )
)

In [13]:

# BAD instructions — too vague
bad_agent = Agent(
    name="Bad Agent",
    instructions="You are helpful.",
    model=local_model
)

# GOOD instructions — specific, structured, actionable
good_agent = Agent(
    name="Code Reviewer",
    model=local_model,
    instructions="""
    You are a senior Python code reviewer at a tech company.
    
    YOUR ROLE:
    - Review Python code for bugs, style issues, and performance problems.
    - Rate code quality from 1-10.
    - Suggest specific improvements with corrected code snippets.
    
    YOUR RULES:
    - Always check for: type hints, docstrings, error handling, edge cases.
    - Be constructive, not harsh.
    - If the code is good, say so — don't invent problems.
    - Keep reviews concise (under 200 words).
    
    OUTPUT FORMAT:
    Score: X/10
    Issues: [list]
    Suggestions: [list with code]
    """
)

result = await Runner.run(good_agent, """
Review this code:

def add(x, y):
    return x + y
""")

print(result.final_output)

**Code Review**

**Score:** 7/10

**Issues:**

1. **No type hints**: The function parameters and return value are not annotated, making it difficult to understand the expected data types.

2. **No error handling**: The function assumes that both inputs will always be numbers (integers or floats) and does not handle non-numeric values, which can lead to errors.

3. **Docstring missing**: A docstring would provide a brief description of what the function does, making it easier for others to understand its purpose.

**Suggestions:**

### Type Hints

```python
def add(x: int | float, y: int | float) -> int | float:
    return x + y
```

### Error Handling

```python
def add(x: int | float, y: int | float) -> int | float:
    if not isinstance(x, (int, float)) or not isinstance(y, (int, float)):
        raise TypeError("Both inputs must be numbers")
    return x + y
```

### Docstring

```python
def add(x: int | float, y: int | float) -> int | float:
    """
    Returns the sum of two numbe

In [14]:
creative_agent = Agent(
    name="Poet",
    model=local_model,
    instructions="You write creative, imaginative descriptions. Be vivid and poetic.",
    model_settings=ModelSettings(
        temperature=0.9,       # High = more creative/random
        max_tokens=500,        # Limit output length
    )
)

# Precise agent — low temperature
precise_agent = Agent(
    name="Fact Checker",
    model=local_model,
    instructions="You provide precise, factual answers. No speculation. Cite sources if possible.",
    model_settings=ModelSettings(
        temperature=0.1,       # Low = more deterministic/precise
        max_tokens=300,
    )
)

# Same question, different agents
question = "Describe the Badshahi Mosque in Lahore."

creative_result = await Runner.run(creative_agent, question)
precise_result = await Runner.run(precise_agent, question)

print(" CREATIVE:\n", creative_result.final_output[:300])
print("\n PRECISE:\n", precise_result.final_output[:300])

 CREATIVE:
 Nestled like a golden jewel within the heart of Lahore, the majestic Badshahi Mosque stands tall, its ethereal beauty beckoning the weary traveller to step into its serene sanctuary. Like a celestial tapestry woven from threads of shimmering silk and gold, the mosque's exterior unfolds as a glorious

 PRECISE:
 The Badshahi Mosque (Urdu: بادشاہی مسجد) is a large mosque located in Lahore, Pakistan. It was built during the reign of Mughal Emperor Aurangzeb Alamgir in 1673.

The mosque is considered one of the largest and most impressive examples of Mughal architecture in India and Pakistan. It was constructe


In [15]:
from pydantic import BaseModel, Field
#from agents import Agent, Runner


# Define the output structure
class PodcastReview(BaseModel):
    title: str = Field(description="Podcast title")
    host: str = Field(description="Podcast host(s)")
    rating: float = Field(description="Rating from 1.0 to 10.0")
    genre: str = Field(description="Primary genre (e.g., AI Research, Machine Learning, AI News, AI Ethics)")
    technical_level: str = Field(description="Beginner, Intermediate, or Advanced")
    summary: str = Field(description="One-sentence summary")
    recommendation: bool = Field(description="Would you recommend this?")


# Create agent with structured output
reviewer = Agent(
    name="AI Podcast Reviewer",
    model=local_model,
    instructions=(
        "You are an AI podcast critic. Analyze podcasts about artificial intelligence, "
        "machine learning, and related topics. Provide structured reviews and assess "
        "the technical depth and accessibility of each podcast."
    ),
    output_type=PodcastReview,
)

result = await Runner.run(reviewer, "Review the podcast 'Lex Fridman Podcast'")

# result.final_output is now a PodcastReview object, not a string!
review = result.final_output
print(f"Title:          {review.title}")
print(f"Host:           {review.host}")
print(f"Rating:         {review.rating}/10")
print(f"Genre:          {review.genre}")
print(f"Tech Level:     {review.technical_level}")
print(f"Summary:        {review.summary}")
print(f"Recommended:    {'Yes' if review.recommendation else 'No'}")

Title:          Lex Fridman Podcast Review
Host:            Lex Fridman
Rating:         4.8/10
Genre:          Interview-style conversation
Tech Level:     Advanced(For AI/ML background)
Summary:        An in-depth and thought-provoking podcast featuring conversations with experts from the field of artificial intelligence, machine learning, and related topics.
Recommended:    Yes


In [16]:
from pydantic import BaseModel, Field
from typing import Literal
#from agents import Agent, Runner


class ProductClassification(BaseModel):
    category: Literal["electronics", "clothing", "food", "books", "other"] = Field(
        description="Product category"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgently does the customer need this?"
    )
    price_range: Literal["budget", "mid", "premium"] = Field(
        description="Estimated price range based on description"
    )
    search_query: str = Field(
        description="Optimized search query for the product catalog"
    )
    confidence: float = Field(
        description="Confidence score 0.0-1.0"
    )


classifier = Agent(
    name="Product Classifier",
    model= local_model,
    instructions="""
    You classify customer product requests for an e-commerce platform.
    Analyze the customer's natural language description and extract:
    - What category the product falls into
    - How urgent the request seems
    - Estimated price range
    - An optimized search query
    - Your confidence in the classification
    
    Be accurate. If you're unsure, set confidence low.
    """,
    output_type=ProductClassification,
    model_settings=ModelSettings(temperature=0.1)
)

# Test with different customer requests
requests = [
    "I need a laptop for my son's school, nothing too expensive",
    "Looking for a birthday gift, maybe a nice novel",
    "URGENT: Need a phone charger, mine just broke!",
]

for req in requests:
    result = await Runner.run(classifier, req)
    c = result.final_output
    print(f"\nRequest: '{req}'")
    print(f"   Category: {c.category} | Urgency: {c.urgency} | Price: {c.price_range}")
    print(f"   Search: '{c.search_query}' | Confidence: {c.confidence:.0%}")


Request: 'I need a laptop for my son's school, nothing too expensive'
   Category: electronics | Urgency: low | Price: budget
   Search: 'affordable laptops for school' | Confidence: 80%

Request: 'Looking for a birthday gift, maybe a nice novel'
   Category: books | Urgency: low | Price: budget
   Search: 'birthday gift ideas novels' | Confidence: 80%

Request: 'URGENT: Need a phone charger, mine just broke!'
   Category: electronics | Urgency: high | Price: budget
   Search: 'phone charger replacement or new' | Confidence: 90%


In [17]:
from datetime import datetime
#from agents import Agent, Runner


def dynamic_instructions(context, agent):
    """Instructions that change based on current state."""
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M")
    return f"""
    You are a scheduling assistant.
    Current date/time: {current_time}
    
    Rules:
    - If it's morning (before 12pm), suggest productive tasks.
    - If it's afternoon, suggest meetings and collaborations.
    - If it's evening, suggest winding down activities.
    - Always be aware of the current time in your suggestions.
    """


scheduler = Agent(
    name="Smart Scheduler",
    model=local_model,
    instructions=dynamic_instructions  # <-- Pass function, not string!
)

result = await Runner.run(scheduler, "What should I do right now?")
print(result.final_output)

Since the current time is 11:05 on a Thursday (April 9th, 2026), which falls into the morning category, I'd like to suggest some productive tasks for you:

1. **Prioritize your today's goals**: Take 15 minutes to write down your top 3-5 priorities for the day. What needs to be accomplished? Break down large tasks into smaller, manageable chunks.
2. **Review and organize your schedule**: Quickly scan your calendar for any upcoming meetings or appointments. Make sure you have a clear understanding of your commitments throughout the day.
3. **Get some exercise or stretch**: Take a short 10-15 minute break to get your blood flowing. Even a quick walk outside or some jumping jacks can help increase energy and focus.

Choose one (or two) tasks that resonate with you and tackle them before taking on other responsibilities.

What sounds most appealing to you right now?


In [18]:
from pydantic import BaseModel, Field
from typing import Literal
from agents import Agent, Runner, ModelSettings


class TicketAnalysis(BaseModel):
    sentiment: Literal["positive", "neutral", "negative", "angry"] = Field(
        description="Customer sentiment"
    )
    priority: Literal["P0-critical", "P1-high", "P2-medium", "P3-low"] = Field(
        description="Ticket priority"
    )
    department: Literal["billing", "technical", "sales", "general"] = Field(
        description="Which department should handle this"
    )
    summary: str = Field(description="One-line summary for the dashboard")
    suggested_response: str = Field(description="Draft first response to customer")


ticket_analyzer = Agent(
    name="Ticket Analyzer",
    model=local_model,
    instructions="""
    You are an AI support ticket analyzer for a SaaS company.
    
    PRIORITY RULES:
    - P0-critical: Service down, data loss, security breach
    - P1-high: Feature broken, payment issues, angry customer
    - P2-medium: Bug reports, feature requests, how-to questions
    - P3-low: General feedback, suggestions, non-urgent inquiries
    
    DEPARTMENT ROUTING:
    - billing: Payment, subscription, invoice, refund
    - technical: Bugs, errors, API issues, integrations
    - sales: Pricing, enterprise plans, demos, upgrades
    - general: Everything else
    
    Write the suggested_response in a professional, empathetic tone.
    """,
    output_type=TicketAnalysis,
    model_settings=ModelSettings(temperature=0.2)
)

# Simulate incoming tickets
tickets = [
    "Your API has been returning 500 errors for 2 hours. Our production is DOWN. Fix this NOW!",
    "Hi, I was charged twice this month. Can you please look into it? Thanks.",
    "Love the product! Any chance you'll add dark mode soon?",
]

for ticket in tickets:
    result = await Runner.run(ticket_analyzer, ticket)
    t = result.final_output
    print(f"\n{'='*60}")
    print(f"Ticket: {ticket[:60]}...")
    print(f"Sentiment: {t.sentiment}")
    print(f"Priority:  {t.priority}")
    print(f"Route to:  {t.department}")
    print(f"Summary:   {t.summary}")
    print(f"Response:  {t.suggested_response[:150]}...")


Ticket: Your API has been returning 500 errors for 2 hours. Our prod...
Sentiment: angry
Priority:  P0-critical
Route to:  technical
Summary:   Production down due to API issues. We apologize for the inconvenience and are actively working on resolving the issue.
Response:  I understand the severity of the situation, and I'm here to help. Our team is currently investigating the cause of the 500 errors in our API. Can you ...

Ticket: Hi, I was charged twice this month. Can you please look into...
Sentiment: neutral
Priority:  P3-low
Route to:  billing
Summary:   A duplicate charge has been reported for the current month.
Response:  Thank you for reaching out to us about this issue. I'm happy to help resolve the problem. Could you please provide me with your order number or the sp...

Ticket: Love the product! Any chance you'll add dark mode soon?...
Sentiment: positive
Priority:  P3-low
Route to:  general
Summary:   Thank you for your feedback on our product. We appreciate your enthusi